# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 8: Model Evaluation & SHAP Explainability
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II  
**Objective**: Evaluate model errors and inspect local/global SHAP values to explain predictions.


In [ ]:
import sys
import warnings
import json
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to system path
sys.path.append(str(Path.cwd().parent))
from src.model_trainer import ModelTrainer
from src.model_evaluator import ModelEvaluator

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'font.size': 11, 'axes.titlesize': 14,
})

DATA_FEATURES = Path.cwd().parent / 'data' / 'features'
MODELS_DIR = Path.cwd().parent / 'models' / 'saved_models'
VIZ_MODEL = Path.cwd().parent / 'visualizations' / 'model'
VIZ_MODEL.mkdir(parents=True, exist_ok=True)


## 1. Load Data and Best Model
We read `model_registry.json` to find the best model and load it.


In [ ]:
# Load features
df_feat = pd.read_csv(DATA_FEATURES / "customer_features.csv")
trainer = ModelTrainer(df_feat, target_col="target_revenue")
X_train, X_test, y_train, y_test, feature_names = trainer.prepare_data(test_size=0.2, random_state=42)

# Read registry
with open(MODELS_DIR.parent / "model_registry.json", "r") as f:
    registry = json.load(f)
    
best_model_name = registry["best_model"]
print(f"Best model based on registry: {best_model_name}")

best_model = joblib.load(MODELS_DIR / f"{best_model_name}.joblib")


## 2. Model Error Plots
Visualizing residual distributions and prediction errors.


In [ ]:
evaluator = ModelEvaluator(y_test, feature_names)
y_pred = best_model.predict(X_test)

# Plot residuals
evaluator.plot_residuals(y_pred, best_model_name, VIZ_MODEL / "residuals.png")

# Plot prediction error
evaluator.plot_prediction_error(y_pred, best_model_name, VIZ_MODEL / "pred_error.png")

print("Saved evaluation charts.")


## 3. Global Explainability via SHAP
Using SHAP values to identify feature importances and their positive/negative directions on predicted future revenue.


In [ ]:
# Run SHAP explanations
evaluator.explain_with_shap(best_model, X_train, X_test, VIZ_MODEL)
